# Stage 10/11 (repurposed) — General F&O trend-following ranking

`score_for_trend_following()` in `strategy_scoring.py` — ranks the
full F&O stock+index 1D universe (see `trend_universe.py` and README
Stage 10 for the repurposing history) for what it's actually for: buy the
share on a fresh UP flip, exit on the opposite DOWN flip, pure cash equity.
Different from the other four scorers in this project — favors a large
favorable move on the UP flips specifically (buying the underlying has no
structural cap the way credit spreads/short IC's containment objective or
debit spread/option buying's ATR-bounded structure do). No stop-loss:
`src/stop_loss_test.py` found no candidate stop rule beats holding to the
natural DOWN-flip exit, so this scores the actual strategy, not a
hypothetical stopped-out version of it.

`security_type` ("Stock"/"Index") is retained purely as a display/filter
column — the ranking is one table across the whole universe, not split by
type (that split existed when the universe included ETFs, which had
structurally smaller typical moves; there are no ETFs in the universe now).

Requires notebook 10 to have been run with real deep history first — if
`up_closed_count` looks thin across the board, that's the symptom of the
now-fixed regression documented in README Stage 10/11 — re-run notebook 10.

In [1]:
import sys
sys.path.append("..")

from src.strategy_scoring import score_for_trend_following, save_ranking

ranked, watchlist = score_for_trend_following()
print(f"Ranked: {len(ranked)}, Watchlist: {len(watchlist)}")
ranked["security_type"].value_counts()

Ranked: 199, Watchlist: 17


security_type
Stock    193
Index      6
Name: count, dtype: int64

## Ranked universe

In [2]:
ranked.reset_index(drop=True)

,symbol,security_type,trend_score,outlier_reliance_flag,gap_risk_flag,up_closed_count,win_rate_up_pct,avg_pct_move_up,median_pct_move_up,median_max_adverse_up
0,BAJFINANCE,Stock,91.6,⚠ RELIES ON OUTLIER WINS (avg-median gap 4.33pp),,39,69.2,12.12,7.79,3.72
1,HDFCBANK,Stock,89.3,,,37,64.9,4.41,3.63,3.25
2,AXISBANK,Stock,87.8,⚠ RELIES ON OUTLIER WINS (avg-median gap 3.40pp),,34,67.6,7.06,3.66,3.70
3,BAJAJHLDNG,Stock,86.9,⚠ RELIES ON OUTLIER WINS (avg-median gap 3.85pp),,30,70.0,7.95,4.10,4.27
4,MIDCPNIFTY,Index,84.9,,,91,65.9,3.39,2.11,0.98
...,...,...,...,...,...,...,...,...,...,...
194,JSWENERGY,Stock,6.4,⚠ RELIES ON OUTLIER WINS (avg-median gap 18.64pp),,38,36.8,16.68,-1.96,7.24
195,POLICYBZR,Stock,6.0,⚠ RELIES ON OUTLIER WINS (avg-median gap 8.30pp),,17,35.3,2.43,-5.87,9.04
196,CDSL,Stock,5.5,⚠ RELIES ON OUTLIER WINS (avg-median gap 15.71pp),,28,39.3,12.22,-3.49,7.38
197,INOXWIND,Stock,5.2,⚠ RELIES ON OUTLIER WINS (avg-median gap 11.98pp),,35,42.9,8.44,-3.54,11.80


In [3]:
ranked[ranked["security_type"] == "ETF"].reset_index(drop=True)

,symbol,security_type,trend_score,outlier_reliance_flag,gap_risk_flag,up_closed_count,win_rate_up_pct,avg_pct_move_up,median_pct_move_up,median_max_adverse_up


In [4]:
# Instruments below the 15 UP-closed-flip floor - not blended into the
# ranking, but not dropped either. Re-check as more calendar time passes.
print(f"On watchlist: {len(watchlist)}")
watchlist.sort_values("avg_pct_move_up", ascending=False).reset_index(drop=True)

On watchlist: 17


,symbol,security_type,up_closed_count,win_rate_up_pct,avg_pct_move_up,median_pct_move_up,median_max_adverse_up
0,ATHERENERG,Stock,2,50.0,48.52,48.52,9.22
1,KAYNES,Stock,11,81.8,30.18,10.86,8.69
2,ETERNAL,Stock,10,50.0,24.82,1.74,6.88
3,PAYTM,Stock,14,57.1,13.88,5.97,7.56
4,KFINTECH,Stock,11,54.5,11.21,2.20,4.90
5,HYUNDAI,Stock,3,33.3,11.12,-4.52,7.99
6,DELHIVERY,Stock,11,63.6,9.22,2.33,5.10
7,IREDA,Stock,10,40.0,8.44,-1.10,6.26
8,WAAREEENER,Stock,6,83.3,5.25,4.64,8.94
9,MANKIND,Stock,11,45.5,4.72,-2.35,4.31
